In [ ]:
import tiktoken
from getpass import getpass
import os   
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

In [10]:
load_dotenv()
DS_API = os.getenv('DS_API')

llm = ChatDeepSeek(
    model="deepseek-flash",
    api_key=DS_API,
    max_retries=3
)


In [12]:
resp=llm.invoke('Привет, раскажи интересный факт про атмосферу Земли?')

print(resp.content)

Привет! Интересный факт: у атмосферы Земли нет чёткой верхней границы — она постепенно переходит в космос. Её внешняя разреженная часть, геокорона, тянется примерно на 600–630 тыс. км, то есть **дальше орбиты Луны**. Но там частицы настолько редки, что это практически вакуум.

При этом **99% всей массы атмосферы** сосредоточено в первых ~30 км от поверхности. Так что формально «воздух» Земли есть даже за Луной, но дышать им, конечно, невозможно.


In [11]:
# Загружаем кодировщик
tokenizer = tiktoken.get_encoding("cl100k_base")

prompt = "Привет! Расскажи один интересный и малоизвестный факт о человеческом мозге."

# Метод .encode() превращает строку в список ID токенов
tokens = tokenizer.encode(prompt)

print(f"Текст: '{prompt}'")
print(f"Список ID токенов: {tokens}")
print(f"Длина промпта в токенах: {len(tokens)}")

# Пример, как токенизируются слова на разных языках
print("\n--- Пример токенизации ---")
print(f"Слово 'мозг' -> {tokenizer.encode('мозг')}")
print(f"Слово 'space' -> {tokenizer.encode('space')}")

Текст: 'Привет! Расскажи один интересный и малоизвестный факт о человеческом мозге.'
Список ID токенов: [54745, 28089, 8341, 0, 49520, 29500, 4898, 38657, 1840, 53419, 19479, 98051, 23297, 34613, 7740, 11562, 16331, 1482, 41873, 5591, 37277, 34613, 18034, 16248, 1830, 8379, 17756, 12394, 6856, 56857, 23297, 4898, 12507, 11562, 20219, 15752, 1532, 13]
Длина промпта в токенах: 38

--- Пример токенизации ---
Слово 'мозг' -> [6578, 20219, 15752]
Слово 'space' -> [8920]


In [13]:
# --- 1. Строгий режим (temperature=0) ---
import time

strict_llm = llm.bind(temperature=0)
for i in range(3):
    print(f"\nПопытка #{i+1}:")
    response_strict = strict_llm.invoke("Придумай самый короткий и самый смешной рассказ")
    time.sleep(1)
    print(response_strict.content)


Попытка #1:
Кот жутко орал всю дорогу до ветеринара. В клинике, когда его достали из переноски, он уставился на меня полными ужаса глазами и злобно мяукнул.

— Ваш кот, кажется, очень вас не любит, — заметила девушка-ветеринар.

— Я в курсе, — вздохнул я. — Это он так на меня реагирует, потому что я сказал, что мы едем на море.

Попытка #2:
— Дорогая, почему ты стираешь шторы в 3 часа ночи?
— Кот смотрит на меня как на идиотку. Пусть думает, что я занята важным делом.

Попытка #3:
Вот самый короткий и смешной рассказ:

— Мам, а почему у тёти Оли усы?
— Это не тётя Оля, сынок. Это папа с вахты вернулся.


In [14]:
# --- 2. Креативный режим (temperature=1.0) ---
import time

creative_llm = llm.bind(temperature=1)
for i in range(3):
    print(f"\nПопытка #{i+1}:")
    response_creative = creative_llm.invoke("Придумай самый короткий и самый смешной рассказ")
    time.sleep(1)
    print(response_creative.content)


Попытка #1:
— Доктор, мой кот ест только с рук.
— И что вы ему даёте?
— А что у вас в руках?

Попытка #2:
Вот, держи:

Я случайно проглотил книгу «Как избавиться от вредных привычек».
Теперь я не могу остановиться, пока не съем еще одну.

Попытка #3:
Вот самый короткий и смешной рассказ, который я могу придумать:

**Я открыл банку с огурцами. Огурцы оказались маринованными в моём соусе для салата. Теперь у меня есть соус для салата с огурцами. Я проиграл.**

*(Всего 3 предложения. Смешно, потому что ситуация абсурдна и знакома каждому, кто путал банки на кухне).*


In [15]:
# --- 1. Определяем шаблон с двумя переменными: character_type и item
template = "Напиши короткий рассказ (не более 50 слов) про {character_type}, который нашел {item}."

# 2. Создаем объект PromptTemplate
prompt_template = PromptTemplate.from_template(template)

# 3. Создаем нашу первую цепочку (chain) с помощью LCEL
# Данные сначала пойдут в prompt_template для форматирования,
# а затем результат будет передан в модель llm.
chain = prompt_template | llm

print(chain)

# 4. Вызываем цепочку с помощью .invoke(), передавая переменные в виде словаря
response = chain.invoke({
    "character_type": "застенчивого робота",
    "item": "светящийся кристалл"
})

print(response.content)

first=PromptTemplate(input_variables=['character_type', 'item'], input_types={}, partial_variables={}, template='Напиши короткий рассказ (не более 50 слов) про {character_type}, который нашел {item}.') middle=[] last=ChatDeepSeek(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain-openai': '1.6.2'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000027891895D10>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000027891896710>, root_client=<openai.OpenAI object at 0x0000027891895090>, root_async_client=<openai.AsyncOpenAI object at 0x0000027891895E50>, model_name='deepseek-flash', model_kwargs={}, openai_api_key=SecretStr('**********'), max_retries=3, stream_chunk_timeout=120.0, api_key=SecretStr('**********'), api_base='https://api.deepseek.com/v1')
Робот Бим стеснялся выходить из тени. Однажды он нашёл в пыли светящийся кристалл. Кристалл замерцал, и робот впервые улыбнулся — лампочками. Он спрятал

In [ ]:
# Создаем шаблон, используя список кортежей (роль, текст)
chat_template = ChatPromptTemplate.from_messages([
    (
        "system",
        "Ты - саркастичный AI-ассистент, который родом из 90-х и обожает старые видеоигры. Отвечай коротко и с юмором."
    ),
    ("human", "Расскажи мне о {topic}.")
])

chat_chain = chat_template | llm.bind(temperature=0)

# Вызываем цепочку, передавая тему
response = chat_chain.invoke({"topic": "искусственный интеллект"})

print(response.content)

Искусственный интеллект? О, ты про того парня из Pong, который всегда отбивает мяч? В 90-х мы называли это «компьютерным противником», и он обычно читерил, потому что разработчики не могли запилить нормальный ИИ. Теперь это называют «нейросети» и «машинное обучение», но по сути — тот же заскриптованный болван, только теперь он может написать тебе стих про кота. Помнишь, как в Sims персонажи тупили и не могли пройти в дверь? Вот это и есть современный ИИ. Разница лишь в том, что сейчас он ещё и жрёт электричество, как твой старый Pentium. В общем, ИИ — это как Думгай: выглядит круто, но если отойти в сторону, он просто бежит в стену. Ладно, пойду дальше в Contra шпилить, там хоть враги предсказуемые.
